# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/pilot/` (local only, gitignored).

In [1]:
import warnings
import plotly.io as pio

from analysis.utils.parser import load_pilot_data
from analysis.pilot.figures import (
    fig_completion_status,
    fig_trial_duration_per_subject,
    fig_moves_per_subject,
    fig_duration_progression,
    fig_moves_progression,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_demographics,
    fig_pairwise_distance_distribution,
)

pio.renderers.default = "browser"

## 0. Load data

In [2]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = load_pilot_data("data/pilot")

df_trials = data["trials"]
df_status = data["status"]

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nTrials dataframe: {df_trials.shape[0]} rows × {df_trials.shape[1]} cols")
print(df_status["completion_status"].value_counts().to_string())

[UserWarning] Participant 6150fbd25056424b64062835: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 69f5f8a6cd820e91eab3f3f7: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 698095549e3f340d0843b024: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 5d9e3ec9611b0b0017b14b9a: revoked consent (status=REJECTED), excluded from trials.
[UserWarning] Participant 69f5bb76f845c6ae7f522328: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 60001f74b9d8d70009041d15: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 67e069199f5b036960b0cc5f: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 694555b4beadda60a5901ec0: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 697cb3

## 1. Completion status

In [3]:
fig_completion_status(df_status).show()

## 2. Trial duration per subject

In [4]:
fig_trial_duration_per_subject(df_trials).show()

## 3. Number of moves per subject

In [5]:
fig_moves_per_subject(df_trials).show()

## 4. Trial duration over task progression

In [6]:
fig_duration_progression(df_trials).show()

## 5. Moves over task progression

In [7]:
fig_moves_progression(df_trials).show()

## 6. Trial duration vs. number of moves

In [8]:
fig_duration_vs_moves(df_trials).show()

## 7. Within-subject variability and reliability

In [9]:
fig_within_subject_variability(df_trials).show()

## 8. Participant demographics

In [10]:
fig_demographics(df_trials).show()

## 9. Pairwise distance distribution

In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

from analysis.pilot.simulate_null_distances import simulate as _sim_null

# Shared null: same K as images_per_trial, enough trials for a stable reference
null_distances = _sim_null(num_dots=20, num_trials=1000, seed=42)
print(f"Null: {len(null_distances):,} distances  mean={null_distances.mean():.3f}  sd={null_distances.std():.3f}")

fig_pairwise_distance_distribution(df_trials, null_distribution=null_distances).show()

Null: 190,000 distances  mean=0.369  sd=0.175


In [12]:
from analysis.pilot.figures import fig_ks_distance_per_subject

# null_distances defined in the cell above (shared across cohorts)
fig_ks_distance_per_subject(df_trials, null_distances).show()

## 10. Temporal engagement

Do subjects work throughout the trial, or front-load moves and sit idle?

- **Left**: cumulative move fraction vs. time fraction within trial. Diagonal = uniform activity; curve bending top-left = front-loading.
- **Right**: average move rate (moves/s, 5 s bins) over absolute time. Dashed line = 60 s (V2 timer unlock).

In [13]:
from analysis.pilot.figures import fig_move_temporal_profile

fig_move_temporal_profile(df_trials).show()

## 11. Reliability (Spearman r) over repeat trials

Styled like the trial-duration/moves progression figures above (sections 4-5): thin
semi-transparent per-subject lines + mean ± SE per version group. Only v3+ subjects have
verbatim trial repeats (v1/v2 have none); v3.x sessions have 3 repeats, v4.x have 4 (2 in the
screening stage, 2 in the experimental stage), so exact sub-versions are pooled into `v3.*`/
`v4.*` groups rather than plotted individually.

In [14]:
from analysis.pilot.figures import fig_reliability_progression

fig_reliability_progression(df_trials).show()

## 12. Cohort comparisons

Pairwise Mann-Whitney U tests between task versions, on per-subject aggregates (one value
per subject, avoiding pseudo-replication across trials). With more than two cohorts present,
every pairwise comparison within a metric is run and p-values are corrected across that
family (default Bonferroni; configurable via `mwu_family(..., correction=...)` — any method
string accepted by `statsmodels.stats.multitest.multipletests` works, e.g. `"holm"`, `"fdr_bh"`).

Five comparisons:
1. Trial duration (s)
2. Moves per trial
3. SNR (σ_d / mean\|Δd\|) — **v1 vs v2 only**, see note before that cell
4. KS distance from null — per-trial D, averaged per subject
5. Idle tail fraction — (RT − t_last_move) / RT

Effect size is rank-biserial *r* = 1 − 2U / (n_A × n_B); \|r\| ≥ 0.5 is conventionally large.

In [15]:
summary = (
    df_trials
    .assign(
        rt_s=df_trials["rt"] / 1000,
        qc_flag_int=df_trials["qc_flag"].astype(int),
    )
    .groupby("task_version")
    .agg(
        n_subjects=("participant_id",  "nunique"),
        n_trials=("trial_number",      "count"),
        rt_s_mean=("rt_s",             "mean"),
        rt_s_median=("rt_s",           "median"),
        rt_s_sd=("rt_s",               "std"),
        rt_s_min=("rt_s",              "min"),
        rt_s_max=("rt_s",              "max"),
        moves_mean=("n_moves",         "mean"),
        moves_median=("n_moves",       "median"),
        moves_sd=("n_moves",           "std"),
        qc_flag_rate=("qc_flag_int",   "mean"),
    )
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary

task_version,v1,v2,v3,v3.06,v4
n_subjects,15.00,10.00,11.00,11.00,3.00
n_trials,150.00,100.00,220.00,220.00,66.00
rt_s_mean,111.19,73.86,78.53,81.83,70.57
rt_s_median,90.15,65.36,64.86,68.58,65.02
rt_s_sd,93.77,22.27,40.64,37.22,21.19
rt_s_min,22.26,60.59,60.04,60.09,60.44
rt_s_max,926.99,227.23,395.10,407.50,222.82
moves_mean,36.11,28.06,28.49,25.37,30.89
moves_median,26.00,27.00,26.50,23.00,31.00
moves_sd,24.02,7.96,12.57,10.18,10.21


In [16]:
import json
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


def mwu_compare(df, value_col, group_a, group_b, group_col="task_version"):
    """Pairwise Mann-Whitney U between two specified groups (no group-count assumption)."""
    a = df.loc[df[group_col] == group_a, value_col].dropna().astype(float).values
    b = df.loc[df[group_col] == group_b, value_col].dropna().astype(float).values
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * U / (len(a) * len(b))
    return {
        "comparison":  f"v{group_a:g} vs v{group_b:g}",
        "N (A)":       len(a),                              "N (B)":       len(b),
        "median (A)":  round(float(np.median(a)), 3),       "median (B)":  round(float(np.median(b)), 3),
        "mean (A)":    round(float(np.mean(a)), 3),         "mean (B)":    round(float(np.mean(b)), 3),
        "std (A)":     round(float(np.std(a, ddof=1)), 3),  "std (B)":     round(float(np.std(b, ddof=1)), 3),
        "U":           round(U, 1),
        "p_raw":       p,
        "r":           round(r, 3),
    }


def mwu_family(df, value_col, group_col="task_version", correction="bonferroni", alpha=0.05):
    """
    Pairwise Mann-Whitney U for every pair of groups in group_col, with a multiple-
    comparison correction applied across the family. `correction` accepts any method
    string supported by statsmodels.stats.multitest.multipletests, e.g. "bonferroni",
    "holm", "fdr_bh", "fdr_by", "sidak", "holm-sidak", ...
    """
    groups = sorted(df[group_col].dropna().unique())
    rows = [mwu_compare(df, value_col, a, b, group_col) for a, b in combinations(groups, 2)]
    result = pd.DataFrame(rows).set_index("comparison")
    _, p_corrected, _, _ = multipletests(result["p_raw"], alpha=alpha, method=correction)
    result["p_raw"] = result["p_raw"].round(4)
    result["p_corrected"] = p_corrected.round(4)
    return result[["N (A)", "N (B)", "median (A)", "median (B)", "mean (A)", "mean (B)",
                    "std (A)", "std (B)", "U", "p_corrected", "p_raw", "r"]]


def print_bottom_line(result, alpha=0.05):
    """One-line plain-English summary per pairwise comparison."""
    for comparison, row in result.iterrows():
        sig = "significant" if row["p_corrected"] < alpha else "not significant"
        print(f"  {comparison}: U={row['U']:.1f}  p_corrected={row['p_corrected']:.4f} ({sig}, "
              f"p_raw={row['p_raw']:.4f})  r={row['r']:.3f}")

### Test 1 — Trial duration

In [17]:
subj_rt = (
    df_trials.assign(rt_s=df_trials["rt"] / 1000)
    .groupby(["participant_id", "task_version"])["rt_s"]
    .mean()
    .reset_index()
)

result_rt = mwu_family(subj_rt, "rt_s", correction="bonferroni")
print_bottom_line(result_rt)
result_rt

  v1 vs v2: U=117.0  p_corrected=0.2133 (not significant, p_raw=0.0213)  r=-0.560
  v1 vs v3: U=116.0  p_corrected=0.8677 (not significant, p_raw=0.0868)  r=-0.406
  v1 vs v3.06: U=113.0  p_corrected=1.0000 (not significant, p_raw=0.1195)  r=-0.370
  v1 vs v4: U=37.0  p_corrected=1.0000 (not significant, p_raw=0.1005)  r=-0.644
  v2 vs v3: U=45.0  p_corrected=1.0000 (not significant, p_raw=0.5035)  r=0.182
  v2 vs v3.06: U=28.0  p_corrected=0.6203 (not significant, p_raw=0.0620)  r=0.491
  v2 vs v4: U=14.0  p_corrected=1.0000 (not significant, p_raw=0.9371)  r=0.067
  v3 vs v3.06: U=39.0  p_corrected=1.0000 (not significant, p_raw=0.1679)  r=0.355
  v3 vs v4: U=19.0  p_corrected=1.0000 (not significant, p_raw=0.7692)  r=-0.152
  v3.06 vs v4: U=29.0  p_corrected=0.6044 (not significant, p_raw=0.0604)  r=-0.758


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,102.269,68.333,111.191,73.856,65.093,14.133,117.0,0.2133,0.0213,-0.560
v1 vs v3,15,11,102.269,70.349,111.191,78.527,65.093,23.771,116.0,0.8677,0.0868,-0.406
v1 vs v3.06,15,11,102.269,80.547,111.191,81.827,65.093,12.863,113.0,1.0000,0.1195,-0.370
v1 vs v4,15,3,102.269,73.489,111.191,70.571,65.093,5.696,37.0,1.0000,0.1005,-0.644
v2 vs v3,10,11,68.333,70.349,73.856,78.527,14.133,23.771,45.0,1.0000,0.5035,0.182
v2 vs v3.06,10,11,68.333,80.547,73.856,81.827,14.133,12.863,28.0,0.6203,0.0620,0.491
v2 vs v4,10,3,68.333,73.489,73.856,70.571,14.133,5.696,14.0,1.0000,0.9371,0.067
v3 vs v3.06,11,11,70.349,80.547,78.527,81.827,23.771,12.863,39.0,1.0000,0.1679,0.355
v3 vs v4,11,3,70.349,73.489,78.527,70.571,23.771,5.696,19.0,1.0000,0.7692,-0.152


### Test 2 — Moves per trial

In [18]:
subj_moves = (
    df_trials
    .groupby(["participant_id", "task_version"])["n_moves"]
    .mean()
    .reset_index()
)

result_moves = mwu_family(subj_moves, "n_moves", correction="bonferroni")
print_bottom_line(result_moves)
result_moves

  v1 vs v2: U=77.0  p_corrected=1.0000 (not significant, p_raw=0.9337)  r=-0.027
  v1 vs v3: U=89.0  p_corrected=1.0000 (not significant, p_raw=0.7555)  r=-0.079
  v1 vs v3.06: U=102.0  p_corrected=1.0000 (not significant, p_raw=0.3241)  r=-0.236
  v1 vs v4: U=21.0  p_corrected=1.0000 (not significant, p_raw=0.9118)  r=0.067
  v2 vs v3: U=57.0  p_corrected=1.0000 (not significant, p_raw=0.9159)  r=-0.036
  v2 vs v3.06: U=68.0  p_corrected=1.0000 (not significant, p_raw=0.3787)  r=-0.236
  v2 vs v4: U=12.0  p_corrected=1.0000 (not significant, p_raw=0.6923)  r=0.200
  v3 vs v3.06: U=73.0  p_corrected=1.0000 (not significant, p_raw=0.4306)  r=-0.207
  v3 vs v4: U=14.0  p_corrected=1.0000 (not significant, p_raw=0.7552)  r=0.152
  v3.06 vs v4: U=11.0  p_corrected=1.0000 (not significant, p_raw=0.4560)  r=0.333


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,27.40,27.700,36.107,28.060,21.355,6.552,77.0,1.0,0.9337,-0.027
v1 vs v3,15,11,27.40,28.800,36.107,28.491,21.355,10.666,89.0,1.0,0.7555,-0.079
v1 vs v3.06,15,11,27.40,24.050,36.107,25.373,21.355,9.061,102.0,1.0,0.3241,-0.236
v1 vs v4,15,3,27.40,33.636,36.107,30.894,21.355,9.682,21.0,1.0,0.9118,0.067
v2 vs v3,10,11,27.70,28.800,28.060,28.491,6.552,10.666,57.0,1.0,0.9159,-0.036
v2 vs v3.06,10,11,27.70,24.050,28.060,25.373,6.552,9.061,68.0,1.0,0.3787,-0.236
v2 vs v4,10,3,27.70,33.636,28.060,30.894,6.552,9.682,12.0,1.0,0.6923,0.200
v3 vs v3.06,11,11,28.80,24.050,28.491,25.373,10.666,9.061,73.0,1.0,0.4306,-0.207
v3 vs v4,11,3,28.80,33.636,28.491,30.894,10.666,9.682,14.0,1.0,0.7552,0.152


### Test 3 — SNR (v1 vs v2 only)

**Why v3 is excluded, not generalized:** v1/v2 reliability is measured from individual image
pairs that incidentally recur across distinct trials (a side effect of the old
`unique_images_per_subject` design — different surrounding images each time). v3 instead
repeats whole trials verbatim (`is_trial_repeat` / `repeat_of_trial_number`), holding context
constant. These two measures convolve different sources of variance — v1/v2 includes context
effects, v3 is closer to pure response noise — so they are not comparable in absolute
magnitude (see the subtitle on the within-subject variability figure above). v3 is therefore
dropped here rather than silently mixed into the v1-vs-v2 comparison.

**For future versions:** if a later version (v4+) also uses the trial-repeat mechanism, add a
separate **Test 3′** comparing it against v3 using the trial-repeat measure
(`_repeated_trial_distances` / `_reliability_pair_distances` in `figures.py`) — do not extend
*this* test to include it.

In [19]:
from collections import defaultdict

from analysis.utils.parser import parse_pairwise_distances


def _subject_snr(df_s):
    """v1/v2-only reliability measure: incidentally-repeated image pairs (see note above)."""
    pair_obs = defaultdict(list)
    for pw_json in df_s["pairwise_distances"]:
        for pair, dist in parse_pairwise_distances(pw_json).items():
            pair_obs[pair].append(dist)
    d1, d2 = [], []
    for obs in pair_obs.values():
        if len(obs) >= 2:
            d1.append(obs[0])
            d2.append(obs[1])
    if not d1:
        return np.nan
    all_dists = [d for pw_json in df_s["pairwise_distances"]
                 for d in parse_pairwise_distances(pw_json).values()]
    sigma_d = float(np.std(all_dists)) if len(all_dists) > 1 else 1.0
    mean_abs_diff = float(np.mean(np.abs(np.array(d1) - np.array(d2))))
    return sigma_d / mean_abs_diff if mean_abs_diff > 0 else np.nan


df_trials_v1v2 = df_trials[df_trials["task_version"].isin([1.0, 2.0])]

subj_snr = (
    df_trials_v1v2
    .groupby(["participant_id", "task_version"])
    .apply(_subject_snr, include_groups=False)
    .reset_index()
    .rename(columns={0: "snr"})
)

result_snr = mwu_family(subj_snr, "snr", correction="bonferroni")
print_bottom_line(result_snr)
result_snr

  v1 vs v2: U=80.0  p_corrected=0.8029 (not significant, p_raw=0.8029)  r=-0.067


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,1.062,1.084,1.089,1.051,0.219,0.16,80.0,0.8029,0.8029,-0.067


### Test 4 — KS distance from null

D is computed per trial against the shared null (see section 9), then averaged per subject —
the same aggregation as `fig_ks_distance_per_subject` above.

In [20]:
from analysis.pilot.figures import trial_ks_distance

# null_distances defined in the figures section above (section 9)
df_trials_ks = df_trials.assign(
    ks_D=df_trials["pairwise_distances"].apply(trial_ks_distance, null_distribution=null_distances)
)
subj_ks = (
    df_trials_ks
    .groupby(["participant_id", "task_version"])["ks_D"]
    .mean()
    .reset_index()
)

result_ks = mwu_family(subj_ks, "ks_D", correction="bonferroni")
print_bottom_line(result_ks)
result_ks

  v1 vs v2: U=42.0  p_corrected=0.7142 (not significant, p_raw=0.0714)  r=0.440
  v1 vs v3: U=79.0  p_corrected=1.0000 (not significant, p_raw=0.8763)  r=0.042
  v1 vs v3.06: U=57.0  p_corrected=1.0000 (not significant, p_raw=0.1945)  r=0.309
  v1 vs v4: U=28.0  p_corrected=1.0000 (not significant, p_raw=0.5735)  r=-0.244
  v2 vs v3: U=82.0  p_corrected=0.6203 (not significant, p_raw=0.0620)  r=-0.491
  v2 vs v3.06: U=68.0  p_corrected=1.0000 (not significant, p_raw=0.3787)  r=-0.236
  v2 vs v4: U=30.0  p_corrected=0.0699 (not significant, p_raw=0.0070)  r=-1.000
  v3 vs v3.06: U=38.0  p_corrected=1.0000 (not significant, p_raw=0.1486)  r=0.372
  v3 vs v4: U=18.0  p_corrected=1.0000 (not significant, p_raw=0.8846)  r=-0.091
  v3.06 vs v4: U=26.0  p_corrected=1.0000 (not significant, p_raw=0.1703)  r=-0.576


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,0.130,0.162,0.146,0.171,0.056,0.044,42.0,0.7142,0.0714,0.440
v1 vs v3,15,11,0.130,0.126,0.146,0.158,0.056,0.069,79.0,1.0000,0.8763,0.042
v1 vs v3.06,15,11,0.130,0.136,0.146,0.187,0.056,0.088,57.0,1.0000,0.1945,0.309
v1 vs v4,15,3,0.130,0.132,0.146,0.120,0.056,0.023,28.0,1.0000,0.5735,-0.244
v2 vs v3,10,11,0.162,0.126,0.171,0.158,0.044,0.069,82.0,0.6203,0.0620,-0.491
v2 vs v3.06,10,11,0.162,0.136,0.171,0.187,0.044,0.088,68.0,1.0000,0.3787,-0.236
v2 vs v4,10,3,0.162,0.132,0.171,0.120,0.044,0.023,30.0,0.0699,0.0070,-1.000
v3 vs v3.06,11,11,0.126,0.136,0.158,0.187,0.069,0.088,38.0,1.0000,0.1486,0.372
v3 vs v4,11,3,0.126,0.132,0.158,0.120,0.069,0.023,18.0,1.0000,0.8846,-0.091


### Test 5 — Idle tail fraction

In [21]:
def _idle_tail_fraction(row):
    """Fraction of trial time after the last move: (RT - t_last) / RT."""
    try:
        moves = json.loads(row["moves"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    ts = [m["t"] for m in moves if isinstance(m.get("t"), (int, float))]
    if not ts or pd.isna(row["rt"]) or row["rt"] <= 0:
        return np.nan
    return (row["rt"] - max(ts)) / row["rt"]


subj_idle = (
    df_trials
    .assign(idle_tail=df_trials.apply(_idle_tail_fraction, axis=1))
    .groupby(["participant_id", "task_version"])["idle_tail"]
    .mean()
    .reset_index()
)

result_idle = mwu_family(subj_idle, "idle_tail", correction="bonferroni")
print_bottom_line(result_idle)
result_idle

  v1 vs v2: U=25.0  p_corrected=0.0604 (not significant, p_raw=0.0060)  r=0.667
  v1 vs v3: U=26.0  p_corrected=0.0366 (significant, p_raw=0.0037)  r=0.685
  v1 vs v3.06: U=49.0  p_corrected=0.8677 (not significant, p_raw=0.0868)  r=0.406
  v1 vs v4: U=0.0  p_corrected=0.0245 (significant, p_raw=0.0025)  r=1.000
  v2 vs v3: U=44.0  p_corrected=1.0000 (not significant, p_raw=0.4597)  r=0.200
  v2 vs v3.06: U=63.0  p_corrected=1.0000 (not significant, p_raw=0.5974)  r=-0.145
  v2 vs v4: U=8.0  p_corrected=1.0000 (not significant, p_raw=0.2867)  r=0.467
  v3 vs v3.06: U=77.0  p_corrected=1.0000 (not significant, p_raw=0.2934)  r=-0.273
  v3 vs v4: U=15.0  p_corrected=1.0000 (not significant, p_raw=0.8846)  r=0.091
  v3.06 vs v4: U=7.0  p_corrected=1.0000 (not significant, p_raw=0.1703)  r=0.576


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,0.039,0.071,0.039,0.115,0.015,0.118,25.0,0.0604,0.0060,0.667
v1 vs v3,15,11,0.039,0.126,0.039,0.193,0.015,0.198,26.0,0.0366,0.0037,0.685
v1 vs v3.06,15,11,0.039,0.058,0.039,0.111,0.015,0.120,49.0,0.8677,0.0868,0.406
v1 vs v4,15,3,0.039,0.114,0.039,0.161,0.015,0.105,0.0,0.0245,0.0025,1.000
v2 vs v3,10,11,0.071,0.126,0.115,0.193,0.118,0.198,44.0,1.0000,0.4597,0.200
v2 vs v3.06,10,11,0.071,0.058,0.115,0.111,0.118,0.120,63.0,1.0000,0.5974,-0.145
v2 vs v4,10,3,0.071,0.114,0.115,0.161,0.118,0.105,8.0,1.0000,0.2867,0.467
v3 vs v3.06,11,11,0.126,0.058,0.193,0.111,0.198,0.120,77.0,1.0000,0.2934,-0.273
v3 vs v4,11,3,0.126,0.114,0.193,0.161,0.198,0.105,15.0,1.0000,0.8846,0.091


## 13. Quality Control

Validate the catch-trial QC thresholds in `SpAM_Task/task_config.json` against actual collected
data. `computeCatchQcFlag` (`utils.js`) flags a catch trial if **any** of:

- `cluster_mean_distance > catch_trials.cluster_max_mean` (0.15) — images not clustered tightly
- `cluster_sd > catch_trials.cluster_max_sd` (0.10) — cluster too spread (SD of pairwise distances; not a stored field, recomputed below)
- `max_dist_to_target > catch_trials.location_tolerance` (0.20) — worst-case per-image distance to the target corner/centre (mirrors `allImagesNearTarget`; not a stored field, recomputed below)

`cluster_mean_distance` is a stored field; `cluster_sd` and `max_dist_to_target` are recomputed
here from `pairwise_distances` / `final_locations` since the task only saves the boolean QC
outcome, not these intermediate values.

**Note**: the `qc_flag` column reflects whichever thresholds were active in `task_config.json`
*at the time each session ran* — which may differ from the current values above (config has
changed over time). The recomputed metrics below are compared against **today's** thresholds,
which is what answers "are these values still valid going forward."

In [22]:
df_catch = data["catch_trials"]

# Current catch_trials QC thresholds (SpAM_Task/task_config.json)
_CATCH_THRESH = {
    "cluster_mean_distance": 0.15,   # catch_trials.cluster_max_mean
    "cluster_sd":            0.10,   # catch_trials.cluster_max_sd
    "max_dist_to_target":    0.20,   # catch_trials.location_tolerance
}

_EDGE = 0.15  # fraction from edge for corner targets (utils.js _targetPoint)
_TARGET_FRAC = {
    "center":              (0.50, 0.50),
    "top left corner":     (_EDGE, _EDGE),
    "top right corner":    (1 - _EDGE, _EDGE),
    "bottom left corner":  (_EDGE, 1 - _EDGE),
    "bottom right corner": (1 - _EDGE, 1 - _EDGE),
}


def _cluster_sd(pw_json):
    """Sample SD (ddof=1) of a trial's pairwise distances -- mirrors computeSD() in utils.js.
    Generic across catch and main trials (same pairwise_distances schema)."""
    dists = list(parse_pairwise_distances(pw_json).values())
    return float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0


def _max_dist_to_target(row):
    """Worst-case (max) per-image normalised distance to the catch target -- mirrors allImagesNearTarget() in utils.js."""
    try:
        locs = json.loads(row["final_locations"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    if not locs:
        return np.nan
    fx, fy = _TARGET_FRAC.get(row["catch_trial_target_location"], (0.5, 0.5))
    w, h = row["sort_area_width"], row["sort_area_height"]
    diag = np.sqrt(w**2 + h**2)
    dists = [
        np.sqrt((loc["x"] * w - fx * w) ** 2 + (loc["y"] * h - fy * h) ** 2) / diag
        for loc in locs
    ]
    return max(dists)


df_catch = df_catch.assign(
    cluster_sd=df_catch["pairwise_distances"].apply(_cluster_sd),
    max_dist_to_target=df_catch.apply(_max_dist_to_target, axis=1),
)


def _qc_row(df, thresh_map, direction="max"):
    """
    Per-cohort QC summary row.

    direction="max": metric is capped (flag if value > thresh); shows median/p90/max
                      -- the upper tail is what's relevant for headroom.
    direction="min": metric is floored (flag if value < thresh); shows median/p10/min
                      -- the lower tail is what's relevant for headroom.
    """
    out = {}
    for metric, thresh in thresh_map.items():
        vals = df[metric].dropna()
        if direction == "max":
            out[(metric, "median")]               = round(float(vals.median()), 3)
            out[(metric, "p90")]                  = round(float(vals.quantile(0.9)), 3)
            out[(metric, "max")]                  = round(float(vals.max()), 3)
            out[(metric, f"flagged (>{thresh})")] = round(float((vals > thresh).mean()), 3)
        else:
            out[(metric, "median")]                = round(float(vals.median()), 3)
            out[(metric, "p10")]                   = round(float(vals.quantile(0.1)), 3)
            out[(metric, "min")]                   = round(float(vals.min()), 3)
            out[(metric, f"flagged (<{thresh})")]  = round(float((vals < thresh).mean()), 3)
    out[("recorded", "qc_flag rate")] = round(float(df["qc_flag"].mean()), 3)
    out[("recorded", "n_trials")]     = len(df)
    return pd.Series(out)


rows = {
    f"v{v:g}": _qc_row(df_catch[df_catch["task_version"] == v], _CATCH_THRESH)
    for v in sorted(df_catch["task_version"].unique())
}
rows["all"] = _qc_row(df_catch, _CATCH_THRESH)

qc_table = pd.DataFrame(rows).T
qc_table.columns = pd.MultiIndex.from_tuples(qc_table.columns)
qc_table

cluster_mean_distance                               cluster_sd         \
                     median    p90    max flagged (>0.15)     median    p90   
v1                    0.045  0.114  0.139             0.0      0.023  0.050   
v2                    0.027  0.046  0.073             0.0      0.014  0.025   
v3                    0.045  0.092  0.113             0.0      0.021  0.041   
v3.06                 0.030  0.094  0.142             0.0      0.013  0.040   
v4                    0.024  0.056  0.062             0.0      0.011  0.027   
all                   0.038  0.093  0.142             0.0      0.018  0.040   

                            max_dist_to_target                               \
         max flagged (>0.1)             median    p90    max flagged (>0.2)   
v1     0.061            0.0              0.115  0.164  0.197            0.0   
v2     0.029            0.0              0.119  0.156  0.183            0.0   
v3     0.049            0.0              0.128  0.162  0.173            0.0   
v3.06  0.079            0.0              0.132  0.173  0.199            0.0   
v4     0.029            0.0              0.133  0.160  0.164            0.0   
all    0.079            0.0              0.127  0.166  0.199            0.0   

          recorded           
      qc_flag rate n_trials  
v1           0.000     30.0  
v2           1.000     20.0  
v3           0.000     44.0  
v3.06        0.000     44.0  
v4           0.000      9.0  
all          0.136    147.0

### Experimental (main) trial thresholds

`computeMainQcFlag` (`utils.js`) flags a main trial if **either**:

- `pairwise_sd < quality_control.min_pairwise_distance_sd` (0.04) — images piled in one spot
- `move_ratio < quality_control.min_move_item_ratio` (0.75), where `move_ratio = n_moves / n_items` — too few moves relative to the number of images on screen

`quality_control.min_trial_rt_ms` (60 s) is excluded here since it's UI-enforced (the Done button
is disabled until the floor elapses), not a post-hoc statistical flag.

Both remaining checks are **floors**, the opposite direction from the catch-trial checks above, so
the table below shows the lower tail (median / p10 / min) and flags values *below* threshold.

In [23]:
# Current main-trial QC thresholds (SpAM_Task/task_config.json)
_MAIN_THRESH = {
    "pairwise_sd": 0.04,   # quality_control.min_pairwise_distance_sd
    "move_ratio":  0.75,   # quality_control.min_move_item_ratio
}


def _n_items(final_locations_json):
    try:
        return len(json.loads(final_locations_json))
    except (json.JSONDecodeError, TypeError):
        return np.nan


df_trials_qc = df_trials.assign(
    pairwise_sd=df_trials["pairwise_distances"].apply(_cluster_sd),
    n_items=df_trials["final_locations"].apply(_n_items),
)
df_trials_qc["move_ratio"] = df_trials_qc["n_moves"] / df_trials_qc["n_items"]

rows = {
    f"v{v:g}": _qc_row(df_trials_qc[df_trials_qc["task_version"] == v], _MAIN_THRESH, direction="min")
    for v in sorted(df_trials_qc["task_version"].unique())
}
rows["all"] = _qc_row(df_trials_qc, _MAIN_THRESH, direction="min")

qc_table_main = pd.DataFrame(rows).T
qc_table_main.columns = pd.MultiIndex.from_tuples(qc_table_main.columns)
qc_table_main

pairwise_sd                               move_ratio               \
           median    p10    min flagged (<0.04)     median    p10   min   
v1          0.186  0.156  0.098             0.0      1.300  0.795  0.45   
v2          0.173  0.144  0.094             0.0      1.350  0.945  0.70   
v3          0.182  0.138  0.088             0.0      1.325  0.750  0.25   
v3.06       0.181  0.130  0.083             0.0      1.150  0.700  0.25   
v4          0.181  0.161  0.135             0.0      1.550  0.925  0.70   
all         0.181  0.142  0.083             0.0      1.300  0.750  0.25   

                          recorded           
      flagged (<0.75) qc_flag rate n_trials  
v1              0.060        0.000    150.0  
v2              0.010        0.000    100.0  
v3              0.091        0.091    220.0  
v3.06           0.132        0.132    220.0  
v4              0.015        0.000     66.0  
all             0.079        0.065    756.0